# SID / Snow.zip → Google Drive

Переливает `Snow.zip` (75.2 ГБ) из Deep Blue Data в папку Google Drive без скачивания на
локальную машину: Globus → диск Colab → распаковка потоком → PNG в JPEG q92 → tar-шарды
по 2 ГБ → Drive. На выходе ~12 ГБ вместо 75.

Источник: <https://deepblue.lib.umich.edu/data/concern/data_sets/cc08hg37c> (CC BY 4.0)

| файл | id | размер |
|---|---|---|
| Clear.zip | `kw52j904w` | 42.5 ГБ |
| Cloudy.zip | `xp68kh09j` | 20.1 ГБ |
| Overcast.zip | `ww72bc35s` | 20.7 ГБ |
| Rain.zip | `qr46r171f` | 35.4 ГБ |
| **Snow.zip** | `8p58pd94g` | **75.2 ГБ** |

## Почему Globus, а не wget

Deep Blue стоит за Cloudflare с JS-челленджем. Проверено: обычный `curl`, `curl` с
браузерными заголовками и `curl_cffi` с подменой TLS-отпечатка Chrome — все получают
`403` и HTML вместо архива. Globus — штатный канал самого репозитория для больших файлов
и Cloudflare не касается.

## Почему шарды, а не голая распаковка

В Snow ~140 тыс. PNG. Запись такого количества мелких файлов в Drive идёт часами и
упирается в rate limit Drive API, а чтение их же на каждой эпохе — ещё медленнее. Один
tar на 2 ГБ пишется и читается на два порядка быстрее и штатно скармливается `webdataset`.

## Порядок

1. ячейка 1 — конфиг
2. ячейка 2 — сколько места на диске Colab (определяет, куда Globus кладёт архив)
3. ячейки 3–4 — Drive: квота, целевая папка, чей объём расходуется
4. ячейки 5–7 — Globus: установка, запуск, перенос
5. ячейки 8–10 — индекс, перепаковка, проверка

**Рантайм — обязательно CPU** (Runtime → Change runtime type → CPU): на GPU-рантайме диск
почти вдвое меньше и 75 ГБ туда не влезут. Обучение — отдельной сессией.

**Папку Drive** сначала открыть по ссылке и нажать «Добавить ярлык на Диск», иначе она не
видна из Colab.


## 1. Конфигурация

In [ ]:

# ---- что качаем -------------------------------------------------------------
FILE_ID     = "8p58pd94g"                           # Snow.zip
DEST_FOLDER = "1UW0HHEvVMnFsi8LsS2PD1dwvMDbiR6Bz"   # id папки Google Drive
PREFIX      = "SID_Snow"                            # префикс имён шардов

# ---- откуда читать архив при перепаковке ------------------------------------
SRC      = "local"                    # "local" — файл принесён Globus; "http" — если ячейка 2 даст OK
ZIP_PATH = "/content/gcp_data/Snow.zip"

# ---- как перепаковываем -----------------------------------------------------
SHARD_BYTES = 2 * 1024**3             # целевой размер шарда
TO_JPEG     = True                    # PNG -> JPEG q=92: ~12 ГБ вместо 75, но lossy
JPEG_Q      = 92
ONLY_SEQ    = None                    # None = всё; иначе список префиксов, напр. ["Snow/Campus_Snow_Day_13"]

# ---- служебное --------------------------------------------------------------
URL  = f"https://deepblue.lib.umich.edu/data/downloads/{FILE_ID}"
UA   = ("Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/537.36 "
        "(KHTML, like Gecko) Chrome/127.0.0.0 Safari/537.36")
WORK = "/content/work"
GB   = 1024**3

import os
os.makedirs(WORK, exist_ok=True)
os.makedirs(os.path.dirname(ZIP_PATH), exist_ok=True)
print("рантайм GPU:", bool(os.environ.get("COLAB_GPU")), "(для этого ноутбука нужен CPU)")

## 2. Место на диске Colab

Snow.zip — 75.2 ГБ. Смотрим, сколько реально свободно: от этого зависит, куда Globus
будет писать архив.

* **свободно > 80 ГБ** → пишем в `/content/gcp_data`, перепаковка читает локальный файл.
  Быстро и без расхода квоты Drive на промежуточный архив.
* **свободно < 80 ГБ** → пишем сразу в примонтированную папку Drive
  (`GCP_TARGET` в ячейке 5), тогда в Drive транзитом ляжет 75 ГБ архива — нужна квота.

In [ ]:

import shutil
t, u, f = shutil.disk_usage("/content")
print(f"диск /content: всего {t/GB:.1f} ГБ, занято {u/GB:.1f} ГБ, свободно {f/GB:.1f} ГБ")
print("нужно под архив: 75.2 ГБ + ~2 ГБ под текущий шард")
print("\nвердикт:", "хватает — Globus пишет на локальный диск"
      if f/GB > 80 else "НЕ хватает — Globus должен писать напрямую в папку Drive")

## 3. Google Drive: квота и целевая папка

In [ ]:

from google.colab import auth, drive as gdrive
auth.authenticate_user()
gdrive.mount("/content/drive")

from googleapiclient.discovery import build
from googleapiclient.http import MediaFileUpload
drive = build("drive", "v3")

q  = drive.about().get(fields="storageQuota,user(emailAddress)").execute()
sq = q["storageQuota"]
used  = int(sq.get("usage", 0)) / GB
limit = int(sq["limit"]) / GB if sq.get("limit") else None
print(f"аккаунт : {q['user']['emailAddress']}")
print(f"занято  : {used:.1f} ГБ" + (f" из {limit:.0f} ГБ  → свободно {limit-used:.1f} ГБ"
                                    if limit else "  (лимит не задан)"))

f_ = drive.files().get(fileId=DEST_FOLDER, supportsAllDrives=True,
                       fields="id,name,driveId,capabilities(canAddChildren)").execute()
shared_drive = "driveId" in f_
print(f"\nпапка   : {f_['name']}")
print("тип     :", "Общий диск — расходуется квота диска, не твоя" if shared_drive
      else "обычная папка в чужом My Drive — файлы лягут в ТВОЮ квоту")
print("запись  :", f_["capabilities"]["canAddChildren"])

need = 12.0 if TO_JPEG else 75.2
print(f"\nнужно под шарды: ~{need:.0f} ГБ" + ("  (+75 ГБ транзитом, если Globus пишет в Drive)"))
if not shared_drive and limit and (limit - used) < need:
    print(f"!! не хватает: свободно {limit-used:.1f} ГБ")

## 4. Чья квота расходуется — проверка

Право записи в папку и расход квоты — разные вещи. Владельцем файла становится тот, кто
его загрузил, и место считается по владельцу, а не по владельцу папки. Исключение —
Общий диск (Shared Drive): там у файлов нет личного владельца, расходуется квота диска.

Ячейка кладёт в целевую папку 10 МБ случайных байт, смотрит `owners` и `quotaBytesUsed`
самого файла и замеряет прирост личной квоты, после чего удаляет пробу. Удаление —
безвозвратное, минуя корзину: файлы в корзине продолжают занимать квоту и исказили бы
замер.

In [ ]:

import io, os, time
from googleapiclient.http import MediaIoBaseUpload

def usage():
    return int(drive.about().get(fields="storageQuota").execute()["storageQuota"].get("usage", 0))

before = usage()
media  = MediaIoBaseUpload(io.BytesIO(os.urandom(10 * 1024**2)),
                           mimetype="application/octet-stream", resumable=False)
probe = drive.files().create(
    media_body=media, supportsAllDrives=True,
    body={"name": "_quota_probe.bin", "parents": [DEST_FOLDER]},
    fields="id,ownedByMe,quotaBytesUsed,owners(emailAddress),driveId").execute()
time.sleep(5)
after = usage()

owner = probe.get("owners", [{}])[0].get("emailAddress", "— (Общий диск, личного владельца нет)")
print("владелец файла       :", owner)
print("ownedByMe            :", probe.get("ownedByMe"))
print("quotaBytesUsed файла :", int(probe.get("quotaBytesUsed", 0)) / 1024**2, "МБ")
print("моя квота выросла на :", (after - before) / 1024**2, "МБ")
print("\nвывод:", "платит Общий диск" if probe.get("driveId")
      else "платите вы — 10 МБ ушло из вашей квоты")

drive.files().delete(fileId=probe["id"], supportsAllDrives=True).execute()
print("проба удалена")

## 5. Globus Connect Personal: установка

Ставится headless, работает из-под NAT — Colab подходит.

Сначала на <https://app.globus.org/file-manager/gcp> завести аккаунт (можно через Google),
назвать эндпоинт, нажать **Generate Setup Key** и вставить ключ в `SETUP_KEY` ниже.

In [ ]:

SETUP_KEY = ""   # <-- вставить ключ с app.globus.org/file-manager/gcp

assert SETUP_KEY, "нужен setup key"

In [ ]:

%%bash
set -e
cd /content
rm -rf /content/gcp globusconnectpersonal-*
wget -q https://downloads.globus.org/globus-connect-personal/linux/stable/globusconnectpersonal-latest.tgz
tar xzf globusconnectpersonal-latest.tgz
mv globusconnectpersonal-*/ /content/gcp      # слэш обязателен: иначе glob цепляет и .tgz
rm -f globusconnectpersonal-latest.tgz
ls /content/gcp/globusconnectpersonal

## 6. Globus Connect Personal: запуск

Два неочевидных момента:

* по умолчанию GCP отдаёт наружу только `$HOME`, а в Colab это `/root` — нужен
  `-restrict-paths`, иначе `/content` не будет виден в File Manager;
* конфигурация живёт в `/root/.globusonline` и умирает вместе с сессией. Копия в Drive
  позволяет после обрыва поднять **тот же** эндпоинт и продолжить перенос, а не заводить
  новый.

In [ ]:

import os, subprocess, shutil, glob, time

GCP        = "/content/gcp/globusconnectpersonal"
GCP_TARGET = "/content/gcp_data"      # <-- если места мало, поменять на путь внутри /content/drive/MyDrive/...
CFG_BACKUP = "/content/drive/MyDrive/globus_colab_cfg"

os.makedirs(GCP_TARGET, exist_ok=True)

# восстановление конфигурации после обрыва сессии
if os.path.isdir(CFG_BACKUP) and not os.path.isdir("/root/.globusonline"):
    shutil.copytree(CFG_BACKUP, "/root/.globusonline")
    print("конфигурация восстановлена из Drive — эндпоинт тот же")
else:
    subprocess.run([GCP, "-setup", "--setup-key", SETUP_KEY], check=True)
    if os.path.isdir(CFG_BACKUP):
        shutil.rmtree(CFG_BACKUP)
    shutil.copytree("/root/.globusonline", CFG_BACKUP)
    print("конфигурация сохранена в Drive")

subprocess.Popen([GCP, "-start", "-restrict-paths", f"rw{GCP_TARGET}"],
                 stdout=open(f"{WORK}/gcp.log", "w"), stderr=subprocess.STDOUT)
time.sleep(15)
print(subprocess.run([GCP, "-status"], capture_output=True, text=True).stdout)
print("отдаётся наружу:", GCP_TARGET)

## 7. Перенос архива

Ячейка ничего не делает — перенос запускается в веб-интерфейсе Globus, дальше он идёт
на серверах Globus и переживает обрывы сессии Colab (пока GCP запущен).

1. На странице датасета нажать **Download Data from Globus** — файлы уже подготовлены
   другим пользователем, коллекция называется `DeepBlueData`.
2. В File Manager включить двухпанельный вид, слева `DeepBlueData` с нужным путём,
   справа — эндпоинт Colab (вкладка **Your Collections**, иконка должна быть зелёной).
3. Справа выбрать путь `GCP_TARGET`, слева отметить `Snow.zip`, нажать **Start**.
4. Прогресс — в разделе **Activity**; на почту придёт письмо по завершении.

Ячейка ниже ждёт появления файла и показывает прогресс.

In [ ]:

import os, time
prev, t0 = 0, time.time()
while True:
    if not os.path.exists(ZIP_PATH):
        print("жду начала переноса..."); time.sleep(30); continue
    sz = os.path.getsize(ZIP_PATH)
    rate = (sz - prev) / 30 / 1024**2
    print(f"{sz/GB:7.2f} ГБ из 75.2  ({100*sz/(75.2*GB):5.1f}%)  {rate:6.1f} МБ/с")
    if sz >= 75.2 * GB * 0.999 and rate == 0:
        print("перенос завершён"); break
    prev = sz
    time.sleep(30)

## 8. Оглавление архива

Нужен `header_offset` каждого файла — по нему делается точное возобновление перепаковки
после обрыва сессии.

In [ ]:
!pip -q install stream-unzip remotezip

In [ ]:

import collections, pickle, zipfile

if SRC == "local":
    with zipfile.ZipFile(ZIP_PATH) as z:
        infos = [(i.filename, i.header_offset, i.file_size)
                 for i in z.infolist() if not i.is_dir()]
else:
    from remotezip import RemoteZip
    with RemoteZip(URL, headers={"User-Agent": UA}) as z:
        infos = [(i.filename, i.header_offset, i.file_size)
                 for i in z.infolist() if not i.is_dir()]

infos.sort(key=lambda t: t[1])
pickle.dump(infos, open(f"{WORK}/infos.pkl", "wb"))

seqs, size = collections.Counter(), collections.Counter()
for n, _, s in infos:
    k = "/".join(n.split("/")[:2]); seqs[k] += 1; size[k] += s

print(f"файлов: {len(infos):,}   распакованный объём: {sum(size.values())/GB:.1f} ГБ\n")
for k in sorted(seqs):
    print(f"{k:<48} {seqs[k]:>8,} файлов   {size[k]/GB:>6.1f} ГБ")

## 9. Перепаковка в шарды

Один последовательный проход `stream-unzip`; каждый распакованный файл кладётся в
открытый tar, шард закрывается на `SHARD_BYTES`, уходит в Drive resumable-загрузкой и
удаляется с диска. Состояние — в `state.json`: при обрыве достаточно перезапустить
ячейку, чтение продолжится с `header_offset` первого необработанного файла.

Про `ONLY_SEQ`: чтение последовательное, поэтому подмножество экономит место в Drive и
время, но не объём чтения — архив всё равно проливается до последнего нужного файла.

При `TO_JPEG=True` узкое место — не диск, а CPU: перекодирование ~140 тыс. кадров 720p
занимает порядка получаса на двух ядрах Colab. Это нормально, ячейка не зависла.

In [ ]:

import io, json, time, tarfile, pickle, requests
from stream_unzip import stream_unzip

infos = pickle.load(open(f"{WORK}/infos.pkl", "rb"))
keep  = {n for n, _, _ in infos if any(n.startswith(p) for p in ONLY_SEQ)} if ONLY_SEQ else None

STATE = f"{WORK}/state.json"
state = json.load(open(STATE)) if os.path.exists(STATE) else {"done": 0, "shard": 0}
print("старт с файла №", state["done"], " шард №", state["shard"])


def zip_bytes(start, chunk=8 * 1024**2):
    """Байты архива начиная со смещения start: локальный файл или HTTP с переподключением."""
    if SRC == "local":
        with open(ZIP_PATH, "rb") as fh:
            fh.seek(start)
            while True:
                c = fh.read(chunk)
                if not c:
                    return
                yield c
    pos = start
    while True:
        try:
            with requests.get(URL, stream=True, timeout=(30, 180),
                              headers={"User-Agent": UA, "Range": f"bytes={pos}-"}) as r:
                r.raise_for_status()
                for c in r.iter_content(chunk):
                    pos += len(c)
                    yield c
            return
        except requests.exceptions.RequestException as e:
            print(f"  обрыв на {pos/GB:.2f} ГБ ({e}) — переподключение"); time.sleep(5)


def upload(path, name):
    media = MediaFileUpload(path, mimetype="application/x-tar",
                            chunksize=64 * 1024**2, resumable=True)
    req = drive.files().create(media_body=media, supportsAllDrives=True,
                               body={"name": name, "parents": [DEST_FOLDER]}, fields="id")
    resp = None
    while resp is None:
        try:
            _, resp = req.next_chunk()
        except Exception as e:
            print("   upload retry:", e); time.sleep(10)
    return resp


def convert(name, data):
    if not (TO_JPEG and name.lower().endswith(".png")):
        return name, data
    from PIL import Image
    buf = io.BytesIO()
    Image.open(io.BytesIO(data)).convert("RGB").save(buf, "JPEG", quality=JPEG_Q, optimize=True)
    return name[:-4] + ".jpg", buf.getvalue()


t0, idx, tar = time.time(), state["done"], None
shard_path = shard_bytes = None

for name, _size, chunks in stream_unzip(zip_bytes(infos[idx][1])):
    name = name.decode("utf-8", "replace")
    data = b"".join(chunks)
    idx += 1
    if name.endswith("/") or (keep is not None and name not in keep):
        continue

    if tar is None:
        shard_path = f"{WORK}/{PREFIX}-{state['shard']:05d}.tar"
        tar, shard_bytes = tarfile.open(shard_path, "w"), 0

    name, data = convert(name, data)
    ti = tarfile.TarInfo(name); ti.size = len(data); ti.mtime = 0
    tar.addfile(ti, io.BytesIO(data))
    shard_bytes += len(data)

    if shard_bytes >= SHARD_BYTES:
        tar.close(); tar = None
        fn = os.path.basename(shard_path)
        upload(shard_path, fn); os.remove(shard_path)
        state.update(done=idx, shard=state["shard"] + 1)
        json.dump(state, open(STATE, "w"))
        read_gb = infos[min(idx, len(infos) - 1)][1] / GB
        print(f"{fn} → Drive | прочитано {read_gb:.1f} ГБ | {read_gb/max(time.time()-t0,1)*3600:.1f} ГБ/ч")

if tar is not None:
    tar.close()
    fn = os.path.basename(shard_path)
    upload(shard_path, fn); os.remove(shard_path)
    state.update(done=idx, shard=state["shard"] + 1)
    json.dump(state, open(STATE, "w"))
    print(f"{fn} → Drive (последний)")

print("готово, шардов:", state["shard"])

## 10. Мелкие файлы, проверка, уборка

In [ ]:

# Readme.txt и метаданные последовательностей лежат в том же архиве Globus рядом со Snow.zip;
# если переносили только Snow.zip, забрать их можно оттуда же вторым transfer-ом.
res = drive.files().list(q=f"'{DEST_FOLDER}' in parents and trashed=false",
                         fields="files(name,size)", pageSize=1000,
                         supportsAllDrives=True, includeItemsFromAllDrives=True).execute()
tot = sum(int(x.get("size", 0)) for x in res["files"])
for x in sorted(res["files"], key=lambda d: d["name"]):
    print(f"{x['name']:<40} {int(x.get('size', 0))/GB:>7.2f} ГБ")
print(f"\nитого {len(res['files'])} файлов, {tot/GB:.1f} ГБ")

# архив больше не нужен
# os.remove(ZIP_PATH)

## 11. Как читать шарды при обучении

```python
import webdataset as wds
ds = wds.WebDataset("/content/drive/MyDrive/<папка>/SID_Snow-{00000..00040}.tar")
```

Шарды читаются потоком, без распаковки на диск. Без webdataset — обычным `tarfile`:

```python
import tarfile, io
from PIL import Image

with tarfile.open(".../SID_Snow-00000.tar") as t:
    for m in t:
        img = Image.open(io.BytesIO(t.extractfile(m).read()))
```
